# StreamFlix Ticket Router — Fine-Tune Qwen3-1.7B-Base with LoRA

**Gen Academy Week 5 custom project** (mirrors the stock `Finetune_Support_Ticket_Classifier_Qwen3.ipynb` phases, with a StreamFlix dataset).

## The Scenario

StreamFlix runs ~30 production microservices owned by 7 platform teams. Every incident
ticket — customer emails, Slack pings, help-portal forms, paraphrased monitoring alerts —
must land in the right **owning-team queue**. Today a frontier model classifies every
ticket; that is slow, expensive, and ships internal ticket text to a third party.

In this notebook you fine-tune **Qwen/Qwen3-1.7B-Base** with **LoRA** (via LLaMA Factory /
LLaMA Board) to route tickets to one of 7 queues:

`Content Discovery · Core Platform · Data Platform · Identity Platform · Monetization Platform · Streaming Platform · User Engagement`

The ground truth comes from the StreamFlix knowledge graph: each ticket concerns a
service, and `service —OWNED_BY_EXTERNAL_TEAM→ team` edges define the owning queue.
The dataset (`streamflix_tickets.csv`, 840 rows) is generated by
`finetuning/generate_tickets.py` in the nexusgraph-ai repo.

> **Runtime**: Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU). Total time ≈ 45–60 min.


## Phase 1 — Check GPU, clone LLaMA-Factory, install

First confirm the runtime has a GPU, then install LLaMA-Factory (~3–5 min).


In [ ]:
# Phase 1a — verify the GPU (expect a Tesla T4 on Colab)
import subprocess

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi not found — are you on a GPU runtime?")

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print("GPU OK:", torch.cuda.get_device_name(0))


In [ ]:
# Phase 1b — clone LLaMA-Factory and install (Colab paths; ~3-5 min)
import os

LF_PARENT = "/content" if os.path.isdir("/content") else os.getcwd()  # guard for local runs
LF_DIR = os.path.join(LF_PARENT, "LLaMA-Factory")

if not os.path.isdir(LF_DIR):
    !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git {LF_DIR}
%cd {LF_DIR}
!pip install -q -e .[torch,bitsandbytes]
!llamafactory-cli version


## Phase 2 — Prepare the StreamFlix ticket dataset

Load `streamflix_tickets.csv` (schema: `category_truth,text`), make a **stratified 80/20
train/validation split**, convert the train rows to **ShareGPT JSON** with a short routing
system prompt, and register the dataset as `streamflix_tickets` in LLaMA-Factory's
`data/dataset_info.json`.

If the CSV is not already present, Colab will prompt you to upload it
(generate it locally with `python finetuning/generate_tickets.py`).


In [ ]:
# Phase 2 — load CSV, split, convert to ShareGPT, register dataset
import json
import os

import pandas as pd
from sklearn.model_selection import train_test_split

QUEUES = [
    "Content Discovery",
    "Core Platform",
    "Data Platform",
    "Identity Platform",
    "Monetization Platform",
    "Streaming Platform",
    "User Engagement",
]
LABEL2ID = {q: i for i, q in enumerate(QUEUES)}

SYSTEM_PROMPT = (
    "You are StreamFlix's incident ticket routing assistant. Given a support ticket, "
    "respond with exactly one of the following owning-team queues: " + ", ".join(QUEUES) + "."
)

# Find the CSV: already on disk (Colab /content or local repo) or upload it.
_candidates = [
    "/content/streamflix_tickets.csv",
    "streamflix_tickets.csv",
    os.path.join(os.path.dirname(os.getcwd()), "streamflix_tickets.csv"),
    "finetuning/streamflix_tickets.csv",
]
csv_path = next((p for p in _candidates if os.path.exists(p)), None)
if csv_path is None:
    try:
        from google.colab import files  # Colab-only

        print("Upload streamflix_tickets.csv now ...")
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except ImportError as exc:  # local run without the file
        raise FileNotFoundError(
            "streamflix_tickets.csv not found. Generate it with finetuning/generate_tickets.py"
        ) from exc

df = pd.read_csv(csv_path)
df = df[df["category_truth"].isin(LABEL2ID)].reset_index(drop=True)
print(f"Loaded {len(df)} tickets from {csv_path}")
print(df["category_truth"].value_counts().to_string())

# Stratified 80/20 split (seeded, reproducible)
df_train, df_val = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["category_truth"]
)
print(f"\nTrain: {len(df_train)}  Validation: {len(df_val)}")

# Convert train rows to ShareGPT messages format
records = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Support Ticket: " + row.text},
            {"role": "assistant", "content": row.category_truth},
        ]
    }
    for row in df_train.itertuples()
]

train_json_path = os.path.join(LF_DIR, "data", "streamflix_train.json")
with open(train_json_path, "w") as fh:
    json.dump(records, fh, indent=2, ensure_ascii=False)
print(f"Wrote {len(records)} ShareGPT records -> {train_json_path}")

# Register the dataset so LLaMA Board can see it
info_path = os.path.join(LF_DIR, "data", "dataset_info.json")
with open(info_path) as fh:
    dataset_info = json.load(fh)
dataset_info["streamflix_tickets"] = {
    "file_name": "streamflix_train.json",
    "formatting": "sharegpt",
    "columns": {"messages": "messages"},
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant",
        "system_tag": "system",
    },
}
with open(info_path, "w") as fh:
    json.dump(dataset_info, fh, indent=2)
print("Registered 'streamflix_tickets' in data/dataset_info.json")

# Keep the validation split for Phase 6
VAL_CSV = os.path.join(LF_PARENT, "val_split.csv")
df_val.to_csv(VAL_CSV, index=False)
print("Validation split saved ->", VAL_CSV)


## Phase 3 — Fine-tune via LLaMA Board (INPUT REQUIRED)

The next cell launches the LLaMA Board web UI. Click the public **gradio.live** link it
prints, then configure the run in the browser:

1. **Model name**: `Qwen/Qwen3-1.7B-Base`  (type it into the model-name box)
2. **Chat template**: `qwen`  (must match — inference below uses the qwen template)
3. **Finetuning method**: `lora` &nbsp; · &nbsp; **Stage**: `Supervised Fine-Tuning`
4. **Dataset**: `streamflix_tickets`  (click *Preview dataset* to sanity-check a few rows)
5. Hyperparameter reference (good T4 defaults):
   | Setting | Value |
   |---|---|
   | Learning rate | `1e-4` |
   | Epochs | `3` |
   | Cutoff length | `1024` |
   | Batch size | `2` |
   | Gradient accumulation | `4` |
   | LoRA rank | `8` |
   | Compute type | `fp16` |
6. Click **Start** and watch the **loss curve** in the UI (~15–25 min for 672 train rows).
7. When it finishes, note the **output dir** (e.g. `train_2026-07-02-...`) shown in the UI —
   Phase 5 will auto-detect it, but write it down in case you need to set it manually.
8. **Stop this cell** (interrupt) once training is done, then continue below.


In [ ]:
# Phase 3 — launch LLaMA Board (leave running while you train; interrupt when done)
%cd {LF_DIR}
!GRADIO_SHARE=1 llamafactory-cli webui


## Phase 4 — Reading the loss curve

Before trusting the adapter, read the training loss curve from LLaMA Board:

- **Healthy**: a steep drop over the first few dozen steps, then a smooth flattening.
  For this task the assistant target is only a 2–3 word queue name, so loss should fall
  from ~2–4 at the start to **well under 0.5** — often near 0.1 — by the end of epoch 3.
- **Flat from the start** (never drops): learning rate too low, wrong dataset selected,
  or a template mismatch (model never sees the labels). Re-check Phase 3 settings.
- **Spiky / diverging**: learning rate too high — halve it and retrain.
- **Instant near-zero loss**: usually label leakage (the answer appears in the prompt) or
  duplicated rows. Our generator asserts no queue name ever appears in ticket text,
  so you should *not* see this.
- A gentle stair-step at each epoch boundary is normal (the sampler reshuffles).

The optional cell below re-plots the curve from `trainer_log.jsonl` so you can screenshot it.


In [ ]:
# Phase 4 (optional) — re-plot the loss curve from trainer_log.jsonl
import glob
import json
import os

logs = sorted(
    glob.glob(os.path.join(LF_DIR, "saves", "**", "trainer_log.jsonl"), recursive=True),
    key=os.path.getmtime,
)
if not logs:
    print("No trainer_log.jsonl found yet — finish a training run in Phase 3 first.")
else:
    log_path = logs[-1]
    steps, losses = [], []
    with open(log_path) as fh:
        for line in fh:
            rec = json.loads(line)
            if "loss" in rec and "current_steps" in rec:
                steps.append(rec["current_steps"])
                losses.append(rec["loss"])
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses)
    plt.xlabel("step")
    plt.ylabel("training loss")
    plt.title("StreamFlix ticket router — LoRA training loss")
    plt.grid(alpha=0.3)
    plt.show()
    print(f"log: {log_path}")
    print(f"start loss: {losses[0]:.3f}   final loss: {losses[-1]:.3f}")
    print("converged" if losses[-1] < 0.5 else "final loss still high — consider more epochs or a higher LR")


## Phase 5 — Merge the LoRA adapter & smoke test (INPUT REQUIRED)

Set `ADAPTER_DIR` to the LLaMA Board output directory (or leave empty to auto-detect the
most recent run), merge the adapter into the base model, define `classify()`, and run a
5-ticket smoke test.


In [ ]:
# Phase 5a — locate the adapter and merge it into the base model
import glob
import os

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

ADAPTER_DIR = ""  # <-- EDIT ME if auto-detect fails, e.g. "/content/LLaMA-Factory/saves/Qwen3-1.7B-Base/lora/train_2026-07-02-12-00-00"
if not ADAPTER_DIR:
    _runs = sorted(
        glob.glob(os.path.join(LF_DIR, "saves", "**", "adapter_config.json"), recursive=True),
        key=os.path.getmtime,
    )
    assert _runs, "No trained adapter found — finish Phase 3, or set ADAPTER_DIR manually."
    ADAPTER_DIR = os.path.dirname(_runs[-1])
print("Using adapter:", ADAPTER_DIR)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
_model = PeftModel.from_pretrained(_base, ADAPTER_DIR).merge_and_unload().eval()

MERGED_DIR = os.path.join(LF_PARENT, "qwen3_streamflix_merged")
_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved ->", MERGED_DIR)


In [ ]:
# Phase 5b — define classify() and run a 5-ticket smoke test
import difflib

import torch


def _qwen_prompt(ticket_text: str) -> str:
    # Must match the 'qwen' template selected in LLaMA Board.
    return (
        "<|im_start|>system\n" + SYSTEM_PROMPT + "<|im_end|>\n"
        "<|im_start|>user\nSupport Ticket: " + ticket_text + "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


@torch.no_grad()
def classify(ticket_text: str) -> str:
    inputs = tokenizer(_qwen_prompt(ticket_text), return_tensors="pt").to(_model.device)
    out = _model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    for q in QUEUES:  # exact/prefix match first
        if text.lower().startswith(q.lower()):
            return q
    return difflib.get_close_matches(text, QUEUES, n=1, cutoff=0.0)[0]  # nearest-label fallback


SMOKE_TESTS = [
    ("Subject: movies keep buffering\n\nHi team, every movie tonight pauses to load every "
     "two minutes and one episode never started at all. My internet is fine. Thanks, Dana",
     "Streaming Platform"),
    ("hey folks, billing-service in prod: duplicate payment captures showing up in the ledger "
     "reconciliation batch again, 218 dupes. who owns this?",
     "Monetization Platform"),
    ("Issue summary: locked out after MFA\nDescription: I am locked out of my account after "
     "the MFA prompt, the verification code is never accepted.\nImpact: Single account affected",
     "Identity Platform"),
    ("Paraphrasing the page we just got: recommendation-service (SEV2) - ranker-v42 inference "
     "errors spiked after the canary promotion, homepage rows falling back to generic.",
     "Content Discovery"),
    ("Hi support, push notifications stopped arriving on my phone entirely this week, but my "
     "tablet on the same account still gets them. Please advise.",
     "User Engagement"),
]

correct = 0
for text, expected in SMOKE_TESTS:
    pred = classify(text)
    ok = pred == expected
    correct += ok
    print(f"[{'OK ' if ok else 'MISS'}] expected={expected:<22} predicted={pred}")
print(f"\nSmoke test: {correct}/5 correct")


## Phase 6 — Evaluate on the validation split

Run the fine-tuned model over the held-out 20% (168 tickets), print the sklearn
`classification_report`, and plot the confusion matrix.

**Routing cost asymmetry**: a mis-routed *Monetization Platform* ticket (a double charge)
costs real refund/chargeback money and trust, while a mis-routed *Content Discovery*
ticket mostly costs a queue-hop. Watch per-class **recall** on the expensive queues,
not just overall accuracy.


In [ ]:
# Phase 6a — fine-tuned model: validation inference + classification report
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

df_val = pd.read_csv(VAL_CSV)
y_true = df_val["category_truth"].tolist()

y_pred = [classify(t) for t in tqdm(df_val["text"], desc="fine-tuned")]

print(classification_report(y_true, y_pred, labels=QUEUES, digits=3, zero_division=0))
ft_acc = accuracy_score(y_true, y_pred)
print(f"Fine-tuned accuracy: {ft_acc:.3f}")


In [ ]:
# Phase 6b — confusion matrix (rows = truth, columns = prediction)
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_true, y_pred, labels=QUEUES)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(QUEUES)), QUEUES, rotation=45, ha="right")
ax.set_yticks(range(len(QUEUES)), QUEUES)
for i in range(len(QUEUES)):
    for j in range(len(QUEUES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("predicted queue")
ax.set_ylabel("true queue")
ax.set_title("StreamFlix ticket router — confusion matrix (counts, color = row recall)")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

for q in ["Monetization Platform", "Identity Platform"]:  # expensive-to-misroute queues
    i = QUEUES.index(q)
    print(f"recall[{q}] = {cm_norm[i, i]:.3f}")


### Baseline vs fine-tuned

**What "baseline" means here**: the *untuned* `Qwen/Qwen3-1.7B-Base` on the exact same
validation tickets. That isolates what the LoRA fine-tune added, holding model size,
tokenizer, and hardware constant.

**Why a letter-choice prompt for the baseline?** A base (non-instruct) model rambles when
asked to emit a multi-word label verbatim, which would unfairly score it near zero. A
constrained multiple-choice prompt — A–G mapped to the 7 queues, "answer with a single
letter" — is the strongest cheap prompt a base model can act on, so the comparison is fair.

**What the delta tells you**: how much task-specific routing knowledge the fine-tune baked
into a 1.7B model that a raw pretrain does not have. A large positive delta on a balanced
7-class task (chance ≈ 14%) is the business case for owning a small router instead of
paying a frontier model per ticket.


In [ ]:
# Phase 6c — baseline: UNTUNED base model with a constrained letter-choice prompt
import re

LETTERS = "ABCDEFG"
LETTER2QUEUE = dict(zip(LETTERS, QUEUES))
_OPTIONS = "\n".join(f"{letter}. {queue}" for letter, queue in LETTER2QUEUE.items())


def _baseline_prompt(ticket_text: str) -> str:
    return (
        "You route StreamFlix support tickets to the owning-team queue.\n\n"
        f"Ticket: {ticket_text}\n\n"
        f"Which queue should handle this ticket?\n{_OPTIONS}\n\n"
        "Answer with a single letter (A-G).\nAnswer:"
    )


# Fresh, untuned copy of the base model (the merged one already contains the adapter).
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
).eval()


@torch.no_grad()
def classify_baseline(ticket_text: str) -> str:
    inputs = tokenizer(_baseline_prompt(ticket_text), return_tensors="pt").to(baseline_model.device)
    out = baseline_model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"[A-G]", text.upper())
    return LETTER2QUEUE[match.group(0)] if match else "UNPARSEABLE"


y_pred_base = [classify_baseline(t) for t in tqdm(df_val["text"], desc="baseline")]

del baseline_model
torch.cuda.empty_cache()

base_acc = accuracy_score(y_true, y_pred_base)
unparseable = sum(p == "UNPARSEABLE" for p in y_pred_base)
print(f"Baseline accuracy (untuned, letter-choice): {base_acc:.3f}  ({unparseable} unparseable)")
print(f"Fine-tuned accuracy:                        {ft_acc:.3f}")
print(f"Accuracy delta (fine-tuned - baseline):     {ft_acc - base_acc:+.3f}")


In [ ]:
# Phase 6d — per-class F1: baseline vs fine-tuned
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score

f1_base = f1_score(y_true, y_pred_base, labels=QUEUES, average=None, zero_division=0)
f1_ft = f1_score(y_true, y_pred, labels=QUEUES, average=None, zero_division=0)

x = np.arange(len(QUEUES) + 1)
width = 0.38
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - width / 2, list(f1_base) + [base_acc], width, label="baseline (untuned)")
ax.bar(x + width / 2, list(f1_ft) + [ft_acc], width, label="fine-tuned (LoRA)")
ax.set_xticks(x, QUEUES + ["overall acc"], rotation=30, ha="right")
ax.set_ylabel("F1 (per class) / accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("StreamFlix ticket router — baseline vs fine-tuned")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Reading the chart: chance on a balanced 7-class task is ~0.14. The gap between the")
print("orange and blue bars is what the LoRA fine-tune added on your own routing taxonomy.")


## Wrap-up

For the Week 5 handout, capture:

1. The **loss curve** screenshot (LLaMA Board or the Phase 4 plot).
2. The Phase 6 **classification report + confusion matrix**.
3. The **baseline vs fine-tuned** chart and the printed accuracy delta.

Paste them into `finetuning/README.md` where the placeholders are.
